<!-- dd:dd-lesson-es-2 -->

# Batch dimensions and applied einsum

*Einsum · `es-2`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# The side panel decides what you practise next. This cell only tells
# the notebook who you are, for the completion beacon.
DD_TOKEN = ""  # paste from the extension's Settings if you want beacons
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"
DD_LESSON_ID = "es-2"


<!-- dd:dd-kp-einsum-batch-dims -->

## Batch dimensions — carrying axes through

`einsum.batch-dims`


### a batch letter runs the contraction per slice


Take a spec you know — matmul is `'ij,jk->ik'` — and prepend one letter to
every part: `'aij,ajk->aik'`. That `a` is a **batch axis**, and here is WHY it
batches:

`j` is shared and dropped, so it's the matmul's contracted axis; `i` and `k`
are its row and column. `a` is different — it appears in **both inputs AND the
output**, and it is never contracted. einsum can't merge anything across `a`,
so it simply *iterates* over it, running the whole `ij,jk->ik` matmul once at
each position of `a`. `a=0` pairs `x[0]` with `u[0]`, `a=1` pairs `x[1]` with
`u[1]`, and so on — independent products, stacked. No loop in your code; the
loop lives in compiled einsum.


In [ ]:
import torch as t

x = t.arange(12.0).reshape(2, 2, 3)     # (a=2, i=2, j=3)
u = t.arange(18.0).reshape(2, 3, 3)     # (a=2, j=3, k=3)

# 'a' everywhere -> run the matmul once per slice of a.
batched = t.einsum('aij,ajk->aik', x, u)
assert batched.shape == (2, 2, 3)
assert t.allclose(batched[0], x[0] @ u[0])   # slice 0 is its own matmul
assert t.allclose(batched[1], x[1] @ u[1])   # slice 1 is its own matmul
print("aij,ajk->aik :", tuple(x.shape), tuple(u.shape), "->",
      tuple(batched.shape))
print(batched)




Why: the per-slice asserts ARE the definition of "batched" — each `batched[k]`
equals `x[k] @ u[k]`. Write exactly that check whenever a batched spec feels
uncertain.


In [ ]:
import torch as t

x = t.arange(12.0).reshape(2, 2, 3)     # (a=2, i=2, j=3)
u = t.arange(18.0).reshape(2, 3, 3)     # (a=2, j=3, k=3)

# 'a' everywhere -> run the matmul once per slice of a.
batched = t.einsum('aij,ajk->aik', x, u)
assert batched.shape == (2, 2, 3)
assert t.allclose(batched[0], x[0] @ u[0])   # slice 0 is its own matmul
assert t.allclose(batched[1], x[1] @ u[1])   # slice 1 is its own matmul
print("aij,ajk->aik :", tuple(x.shape), tuple(u.shape), "->",
      tuple(batched.shape))
print(batched)


<!-- dd:dd-q268 -->

### Problem 268 · faded

Apply the *same* batching idea to matrix-vector products: a batch of matrices
`(b, i, j)` times a batch of vectors `(b, j)`, item n being `a[n] @ x[n]`.
(Start from matvec `'ij,j->i'` — where does the batch letter go?)


In [ ]:
import torch as t

def solve(a, x):
    """(b,i,j) x (b,j) -> (b,i): each matrix times ITS OWN vector."""
    return t.einsum('_____', a, x)


### omit the batch letter from one input — sharing


Now change one thing: leave the batch letter OFF one operand. `'bij,jk->bik'`
— the first factor has `b`, the second (`jk`) does not. WHY this shares:

The second operand has no `b`-axis, so it has nothing to vary as `b` moves —
einsum reuses that *same* matrix at every batch position. The base matmul
`ij,jk->ik` still runs per item, but the second factor is a single **constant**
applied to all of them. A missing batch letter means "broadcast this operand
across the batch" — no `t.tile`, no loop, just its absence.


In [ ]:
import torch as t

x = t.arange(12.0).reshape(2, 2, 3)     # (b=2, i=2, j=3): a batch
m = t.arange(9.0).reshape(3, 3)         # ONE (3,3) matrix — no batch axis

# 'm' has no b -> the same m multiplies every item of the batch.
shared = t.einsum('bij,jk->bik', x, m)
assert t.allclose(shared[0], x[0] @ m)
assert t.allclose(shared[1], x[1] @ m)   # same m both times
print("m has no b, so one matrix serves the whole batch:",
      tuple(m.shape), "->", tuple(shared.shape))
print(shared)




Why: notice what you did NOT write — no tiling of `m`, no broadcasting
gymnastics. Omitting `b` from the second operand IS the sharing.


In [ ]:
import torch as t

x = t.arange(12.0).reshape(2, 2, 3)     # (b=2, i=2, j=3): a batch
m = t.arange(9.0).reshape(3, 3)         # ONE (3,3) matrix — no batch axis

# 'm' has no b -> the same m multiplies every item of the batch.
shared = t.einsum('bij,jk->bik', x, m)
assert t.allclose(shared[0], x[0] @ m)
assert t.allclose(shared[1], x[1] @ m)   # same m both times
print("m has no b, so one matrix serves the whole batch:",
      tuple(m.shape), "->", tuple(shared.shape))
print(shared)


<!-- dd:dd-q279 -->

### Problem 279 · faded

A linear layer without bias: a batch of vectors `x` of shape `(b, d)` times
ONE transformation matrix `w` of shape `(d, e)`, giving `(b, e)`. (Which
operand should be missing the batch letter?)


In [ ]:
import torch as t

def solve(x, w):
    """(b,d) batch of vectors, ONE (d,e) matrix w -> (b,e)."""
    return t.einsum('_____', x, w)


### drop the batch letter from the OUTPUT — summing over the batch


Third variation: keep the batch letter in both inputs, but leave it OUT of the
output. `'bi,bj->ij'` — `b` is in both inputs, absent from `ij`. WHY this sums:

Any letter dropped from the output is summed. `b` is shared across the inputs
(so the per-item outer products are formed), but dropping it from the output
collapses them: you get **Σ over the batch** of `outer(v_b, v_b)`, one `(i, j)`
matrix — not a `(b, i, j)` stack. Dropping the batch letter is how "sum over
the dataset" enters a spec.


In [ ]:
import torch as t

v = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])               # (b=2, i=2): a batch of vectors

# 'b' in both inputs, absent from output -> outer products SUMMED over b.
gram = t.einsum('bi,bj->ij', v, v)
by_hand = t.outer(v[0], v[0]) + t.outer(v[1], v[1])
assert t.allclose(gram, by_hand)
print("b is in the inputs but not the output, so it is SUMMED away:")
print(gram)
print("outer(v0,v0) + outer(v1,v1) agrees:", bool(t.allclose(gram, by_hand)))




Why: two readings must agree — algebraically it's `Σ_b outer(v_b, v_b)`;
mechanically `b` is shared (multiplied along) and absent from the output
(summed). When both readings match, the spec is right.


In [ ]:
import torch as t

v = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])               # (b=2, i=2): a batch of vectors

# 'b' in both inputs, absent from output -> outer products SUMMED over b.
gram = t.einsum('bi,bj->ij', v, v)
by_hand = t.outer(v[0], v[0]) + t.outer(v[1], v[1])
assert t.allclose(gram, by_hand)
print("b is in the inputs but not the output, so it is SUMMED away:")
print(gram)
print("outer(v0,v0) + outer(v1,v1) agrees:", bool(t.allclose(gram, by_hand)))


<!-- dd:dd-q310 -->

### Problem 310 · faded

A batch of vectors `v` of shape `(b, d)` → the SINGLE `(d, d)` matrix that sums
every vector's self outer product. (The batch letter is in both inputs. Where
must it NOT appear, so that the per-item matrices collapse into one?)


In [ ]:
import torch as t

def solve(v):
    """(b,d) -> (d,d): sum over the batch of outer(v[n], v[n])."""
    return t.einsum('_____', v, v)


<!-- dd:dd-q250 -->

### Problem 250 · guided

Write a function solve(a) that takes a 3-D tensor of shape (b, i, d) — a batch of i feature vectors of dimension d — and returns the (b, i, i) batch of similarity matrices: entry [n, p, q] is the dot product of vectors p and q within batch item n.


<details>
<summary>Hints</summary>

1. Every batch item needs its OWN similarity matrix, so the batch index
   survives to the output — it is not one of the summed indices.
2. You need two views of the same tensor: one indexed `b i d`, one `b j
   d`. Which single letter is missing from the output side?
3. `t.einsum('bid,bjd->bij', a, a)` — `d` is contracted, `b` is carried,
   `i` and `j` are the two vector slots.

</details>


In [ ]:
import torch as t

def solve(a):
    """Return the (b, i, i) batch of similarity matrices."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 2, 2)))


<!-- dd:dd-q265 -->

### Problem 265 · independent

Write a function solve(a, u) that takes two batches of matrices — a of shape (a, i, j) and u of shape (a, j, k) — and returns the (a, i, k) batch where each item is the matrix product a[n] @ u[n].


In [ ]:
import torch as t

def solve(a, u):
    """Return the (a, i, k) batch where each item is the matrix product a[n] @ u[n]."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 2, 2)), t.ones((1, 2, 2))))


<!-- dd:dd-q276 -->

### Problem 276 · independent

Write a function solve(a, b) that takes two 3-D tensors a of shape (b, i, k) and b of shape (b, k, j), and returns the (b, i, j) batched matrix product — contracting the shared inner axis independently for each leading batch index.


In [ ]:
import torch as t

def solve(a, b):
    """Return the (b, i, j) batched matrix product — contracting the shared inner axis independently for each leading"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 1, 2)), t.ones((2, 2, 1))))


<!-- dd:dd-q297 -->

### Problem 297 · independent

Write a function solve(b_stack, m) that takes a batch of matrices of shape (b, i, j) and ONE constant matrix m of shape (j, k), and returns the (b, i, k) batch multiplying every item by the same m.


In [ ]:
import torch as t

def solve(b_stack, m):
    """Return the (b, i, k) batch multiplying every item by the same m."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 1, 2)), t.ones((2, 3))))


<!-- dd:dd-q243 -->

### Problem 243 · independent

Write a function solve(u, v) that takes two batches of vectors — u of shape (b, i) and v of shape (b, j) — and returns the batch of OUTER PRODUCTS: a (b, i, j) tensor where entry [n] is the outer product of u[n] with v[n] (every element of u[n] times every element of v[n]). Einsum states it directly; broadcasting works too.


In [ ]:
import torch as t

def solve(u, v):
    """Return the per-batch outer products of u and v."""
    return None


# Example run — the grader calls solve() with several batches.
print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0, 4.0, 5.0]])))


<!-- dd:dd-q287 -->

### Problem 287 · independent

Write a function solve(x) that takes a batch of vectors of shape (b, d) and returns the (b, d, d) batch of SELF outer products: item n is the matrix x[n, i] * x[n, j] of all pairwise feature interactions within vector n.


In [ ]:
import torch as t

def solve(x):
    """Return the (b, d, d) batch of SELF outer products."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0]])))


<!-- dd:dd-q284 -->

### Problem 284 · independent

Write a function solve(v) that takes a batch of vectors of shape (b, d) and returns a length-b tensor where entry n is the SUM OF SQUARES of vector n's elements (its squared Euclidean norm).


In [ ]:
import torch as t

def solve(v):
    """Return a length-b array where entry n is the SUM OF SQUARES of vector n's elements (its squared Euclidean norm"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[3.0, 4.0]])))


#### Common mistakes

- **"Batching needs a loop or `vmap`."** — One letter, present
  everywhere, batches any contraction. The loop exists only in compiled code.
- **"All operands must mention every letter."** — An operand OMITTING the
  batch letter is broadcast across the batch — that's the shared case, not an
  error. Only the OUTPUT omitting it changes the math, by summing.
- **The three fates of a batch letter** — the whole content of batching, told
  apart by where the letter appears: in both inputs *and* the output
  (`bik,bkj->bij`) = **parallel**, independent per-item products; in one input
  only (`bij,jk->bik`) = **shared**, one constant applied to every item; in
  both inputs but dropped from the output (`bi,bj->ij`) = **summed** over the
  batch. One letter's placement is the difference between per-sample results
  and a dataset aggregate.


<!-- dd:dd-kp-einsum-broadcast-scaling -->

## Weighted sums and per-axis scaling

`einsum.broadcast-scaling`


A hugely common tensor operation: **one small vector of weights applied
along one axis of a big tensor** — per-channel scales, per-timestep weights,
head-importance vectors. In einsum, the vector simply declares WHICH axis it
rides on, by using that axis's letter; the output side then decides between
the operation's two flavors:

- **Weighted SUM (collapse the axis):**
  `'chw,c->hw'` — the image's channel axis pairs with the weight vector,
  and c is dropped: each output pixel is Σ_c w[c]·img[c,h,w]. A grayscale
  conversion is exactly this. Likewise `'btd,t->bd'` (weights per timestep,
  summed over time), `'btd,d->bt'` (project features onto a vector).
- **Weighted SCALE (keep the axis):**
  `'chw,c->chw'` — same pairing, but c survives: channel k is multiplied
  by s[k], nothing summed. The broadcasting equivalent is
  `img * s[:, None, None]` — einsum saves you counting the Nones, because
  the LETTER finds the axis by name.

The recipe for any such task: (1) letter the big tensor meaningfully,
(2) give the vector the letter of the axis it describes, (3) keep or drop
that letter per whether the task says "scale/weight each…" (keep) or
"weighted sum/average over…" (drop). A weighted AVERAGE divides by
`w.sum()` outside the spec (einsum never divides — same as means).

These little specs are constant companions in model code — attention-head
weighting `'bthd,h->btd'` is exactly the same shape of thought as grayscale
conversion.


Task: weighted sum over channels (grayscale-style), and per-channel scaling
— same operands, one letter's fate apart.


In [ ]:
import torch as t

img = t.arange(12.0).reshape(3, 2, 2)     # (c=3, h=2, w=2)
w = t.tensor([0.5, 0.25, 0.25])            # one weight per channel

# COLLAPSE: c pairs with the weights and is dropped -> weighted sum.
gray = t.einsum('chw,c->hw', img, w)
assert gray.shape == (2, 2)
# pixel (0,0): 0.5*img[0,0,0] + 0.25*img[1,0,0] + 0.25*img[2,0,0]
assert gray[0, 0] == 0.5 * 0.0 + 0.25 * 4.0 + 0.25 * 8.0

# KEEP: same pairing, c survives -> per-channel scaling.
s = t.tensor([1.0, 10.0, 100.0])
scaled = t.einsum('chw,c->chw', img, s)
assert scaled.shape == img.shape
assert scaled[1, 0, 0] == img[1, 0, 0] * 10.0     # channel 1 scaled by 10
# Broadcasting twin — the letters replaced None-counting:
assert t.allclose(scaled, img * s[:, None, None])

# Weighted AVERAGE of rows: weighted sum / total weight — divide outside.
a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
wr = t.tensor([3.0, 1.0])
wavg = t.einsum('n,nd->d', wr, a) / wr.sum()
assert t.allclose(wavg, t.tensor([1.5, 2.5]))
print("chw,c->hw  (c dropped)  ", tuple(gray.shape))
print(gray)
print("chw,c->chw (c kept)     ", tuple(scaled.shape),
      "channel 1 scaled by 10:", scaled[1, 0, 0].item())
print("weighted average with weights", wr, ":", wavg)




Why each step:

1. The two specs differ ONLY in whether c appears after `->` — pausing on
   that pair is the fastest way to internalize keep-vs-drop as the
   sum-vs-scale switch.
2. The hand-computed pixel check is the standard verification for weighted
   ops: pick one output element, expand its formula, compare. One element
   suffices — the spec treats all positions identically.
3. In the weighted average, einsum handles the numerator only. The
   denominator (`w.sum()`) lives outside — remembering WHERE the division
   goes is most of q255.


In [ ]:
import torch as t

img = t.arange(12.0).reshape(3, 2, 2)     # (c=3, h=2, w=2)
w = t.tensor([0.5, 0.25, 0.25])            # one weight per channel

# COLLAPSE: c pairs with the weights and is dropped -> weighted sum.
gray = t.einsum('chw,c->hw', img, w)
assert gray.shape == (2, 2)
# pixel (0,0): 0.5*img[0,0,0] + 0.25*img[1,0,0] + 0.25*img[2,0,0]
assert gray[0, 0] == 0.5 * 0.0 + 0.25 * 4.0 + 0.25 * 8.0

# KEEP: same pairing, c survives -> per-channel scaling.
s = t.tensor([1.0, 10.0, 100.0])
scaled = t.einsum('chw,c->chw', img, s)
assert scaled.shape == img.shape
assert scaled[1, 0, 0] == img[1, 0, 0] * 10.0     # channel 1 scaled by 10
# Broadcasting twin — the letters replaced None-counting:
assert t.allclose(scaled, img * s[:, None, None])

# Weighted AVERAGE of rows: weighted sum / total weight — divide outside.
a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
wr = t.tensor([3.0, 1.0])
wavg = t.einsum('n,nd->d', wr, a) / wr.sum()
assert t.allclose(wavg, t.tensor([1.5, 2.5]))
print("chw,c->hw  (c dropped)  ", tuple(gray.shape))
print(gray)
print("chw,c->chw (c kept)     ", tuple(scaled.shape),
      "channel 1 scaled by 10:", scaled[1, 0, 0].item())
print("weighted average with weights", wr, ":", wavg)


<!-- dd:dd-q269 -->

### Problem 269 · faded

Weighted sum over channels: (c,h,w) and weights (c,) → (h,w).


In [ ]:
import torch as t

def solve(img, w):
    """Per-pixel weighted sum across channels."""
    return t.einsum('_____', img, w)


<!-- dd:dd-q291 -->

### Problem 291 · guided

Write a function solve(img, s) that takes a channels-first image of shape (c, h, w) and a per-channel scale vector of length c, and returns the image with each channel MULTIPLIED by its scale — same shape (c, h, w), no summation at all.


<details>
<summary>Hints</summary>

1. Scale each channel by s[k], image shape unchanged — is the channel letter
   kept or dropped?
2. Same left-hand side as the weighted sum; the output keeps everything.
3. `'chw,c->chw'` — check one entry of a scaled channel.

</details>


In [ ]:
import torch as t

def solve(img, s):
    """Return the image with each channel MULTIPLIED by its scale — same shape (c, h, w), no summation at all."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 1, 2)), t.tensor([2.0, 0.5])))


<!-- dd:dd-q251 -->

### Problem 251 · independent

Write a function solve(x, v) that takes a 3-D tensor x of shape (b, t, d) — a batch of t tokens with d features — and a 1-D vector v of length d, and returns the (b, t) tensor where each token's feature vector has been reduced to its dot product with v.


In [ ]:
import torch as t

def solve(x, v):
    """Return the (b, t) array where each token's feature vector has been reduced to its dot product with v."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 2, 3)), t.tensor([1.0, 2.0, 3.0])))


<!-- dd:dd-q304 -->

### Problem 304 · independent

Write a function solve(x, w) that takes a tensor x of shape (b, t, d) and a weight vector w of length t — one weight per TIME position — and returns the (b, d) weighted sum over the time axis: each token's features scaled by its position's weight, then summed across t. (A companion drill weights the FEATURE axis; here the contraction letter is t.)


In [ ]:
import torch as t

def solve(x, w):
    """Return the (b, d) weighted sum over the time axis."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 2, 2)), t.tensor([1.0, 3.0])))


<!-- dd:dd-q309 -->

### Problem 309 · independent

Write a function solve(x, w) that takes a multi-head tensor x of shape (b, t, h, d) and a head-importance vector w of length h, and returns the (b, t, d) tensor mixing the heads: at each position, the w-weighted sum of the h head outputs.


In [ ]:
import torch as t

def solve(x, w):
    """Return the (b, t, d) tensor mixing the heads."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 1, 2, 2)), t.tensor([0.5, 0.5])))


<!-- dd:dd-q255 -->

### Problem 255 · independent

Write a function solve(a, w) that takes a 2-D tensor a of shape (n, d) and a 1-D weight vector w of length n, and returns the WEIGHTED AVERAGE of a's rows as a length-d vector: the w-weighted sum of the rows divided by the total weight (assume w.sum() is nonzero).


In [ ]:
import torch as t

def solve(a, w):
    """Return the WEIGHTED AVERAGE of a's rows as a length-d vector."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[0.0, 0.0], [10.0, 20.0]]), t.tensor([1.0, 3.0])))


<!-- dd:dd-q253 -->

### Problem 253 · independent

Write a function solve(f, g) that takes two 3-D tensors of identical shape (c, h, w) — c channels of h x w maps — and returns the (h, w) tensor formed by multiplying corresponding elements and summing across the CHANNEL dimension only.


In [ ]:
import torch as t

def solve(f, g):
    """Return the (h, w) array formed by multiplying corresponding elements and summing across the CHANNEL dimension """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 2, 2)), t.ones((2, 2, 2))))


#### Common mistakes

- **"The weight vector needs reshaping to match the tensor."** — In einsum
  the LETTER aligns it; `'chw,c->…'` is the whole alignment. Reshaping with
  None is the broadcasting spelling of the same thing — fine too, but count
  your Nones.
- **"Scale vs weighted-sum are different kinds of operation."** — Same
  multiply along the paired axis; the output side's keep/drop is the only
  difference. Task words map directly: "scale/weight each X" → keep;
  "combine/sum/average over X" → drop.
- **"einsum can compute the weighted average directly."** — No division in
  the notation. Weighted sum inside, `/ w.sum()` outside — and use the
  WEIGHTS' total, not the item count.


<!-- dd:dd-kp-einsum-attention-patterns -->

## Attention-shaped contractions

`einsum.attention-patterns`


Transformer attention is two einsums with a softmax between them — and both
einsums are patterns you already own, at scale. The tensors:
queries/keys/values shaped (batch, heads, sequence, features) = `bhqd` /
`bhkd` / `bhkd`, where q and k index query/key positions and d is the
feature dimension.

**Scores: "every query against every key."**
`'bhqd,bhkd->bhqk'` — read it with the rules: b, h batch (everywhere);
d shared+dropped (dot product); q and k each private+kept (all pairs).
So within each (batch, head): the (q, k) table of query·key dots — a
batched pairwise-similarity matrix, precisely np-4's pairwise pattern in
einsum clothes. The single-head version drops the h: `'bqd,bkd->bqk'`.

**Mixing: "weighted average of values."**
`'bhqk,bhkd->bhqd'` — k shared+dropped: for each query, sum the value
vectors weighted by its attention row. Output back to (b, h, q, d).

Around these live the **projection** patterns — a weight matrix applied to
the last axis of a sequence tensor:

- `'btc,cf->btf'` (or `'bth,hd->btd'`) — shared weights across batch AND
  time: the batch-dims "shared operand" case with two carried letters.
- Stacked projections `'btd,nde->nbte'` — n projection matrices applied at
  once, n surviving as a new leading axis.
- Chains: `'bi,ih,ho->bo'` — a two-layer linear network in one spec (no
  nonlinearity, of course — einsum is linear algebra only).

Nothing here is new machinery. The exercise of this KP is READING these
five-letter specs fluently: batch letters ride, one letter contracts, the
rest position the output. When you can gloss `'bhqk,bhkd->bhqd'` as
"attention-weighted sum of values" at sight, this lesson has done its job.


Task: single-head attention scores, then the weighted value mix — on tiny
tensors where every number is checkable.


In [ ]:
import torch as t

# One batch item, sequences of 2 queries / 2 keys, feature dim 3.
q = t.tensor([[[1.0, 0.0, 0.0],
               [0.0, 1.0, 0.0]]])          # (b=1, q=2, d=3)
k = t.tensor([[[1.0, 0.0, 0.0],
               [1.0, 1.0, 0.0]]])          # (b=1, k=2, d=3)

# Scores: d contracts; q and k combine -> (1, 2, 2) table of dots.
scores = t.einsum('bqd,bkd->bqk', q, k)
assert scores.shape == (1, 2, 2)
assert scores[0].tolist() == [[1.0, 1.0],    # q0.k0, q0.k1
                              [0.0, 1.0]]    # q1.k0, q1.k1

# (Real attention: scale by 1/sqrt(d), softmax over k. Both live OUTSIDE
# einsum — they're not contractions.)
s = t.tensor([[[0.5, 0.5],
               [0.0, 1.0]]])               # a fake attention matrix (b,q,k)
v = t.tensor([[[10.0, 0.0],
               [0.0, 10.0]]])              # (b, k, d2=2)

# Mixing: k contracts -> each query gets its weighted sum of value rows.
out = t.einsum('bqk,bkd->bqd', s, v)
assert out[0, 0].tolist() == [5.0, 5.0]     # 0.5*v0 + 0.5*v1
assert out[0, 1].tolist() == [0.0, 10.0]    # 1.0*v1 — query 1 attends key 1

# Projection: one (d2, f) matrix shared across batch and sequence.
w = t.tensor([[1.0, 0.0, 0.0],
              [0.0, 1.0, 1.0]])            # (d2=2, f=3)
proj = t.einsum('bqd,df->bqf', out, w)
assert proj.shape == (1, 2, 3)
assert proj[0, 1].tolist() == [0.0, 10.0, 10.0]
print("bqd,bkd->bqk  scores", tuple(scores.shape))
print(scores[0])
print("bqk,bkd->bqd  mixed ", tuple(out.shape))
print(out[0])
print("bqd,df->bqf   proj  ", tuple(proj.shape))
print(proj[0])




Why each step:

1. Unit-vector queries make the score table transparent: q0 = e₁ dots both
   keys to 1; q1 = e₂ only overlaps the second key. Verify pairwise specs
   on bases first, random data second.
2. The mixing step's per-query reading ("query 1 puts all weight on key 1 →
   gets value row 1") is the semantic gloss of `'bqk,bkd->bqd'` — attach
   meanings to letters and the spec narrates itself.
3. The projection reuses the shared-operand batching: w carries no b or q,
   so one matrix serves every position. Full attention = these three
   contractions + softmax; you have now executed each part.


In [ ]:
import torch as t

# One batch item, sequences of 2 queries / 2 keys, feature dim 3.
q = t.tensor([[[1.0, 0.0, 0.0],
               [0.0, 1.0, 0.0]]])          # (b=1, q=2, d=3)
k = t.tensor([[[1.0, 0.0, 0.0],
               [1.0, 1.0, 0.0]]])          # (b=1, k=2, d=3)

# Scores: d contracts; q and k combine -> (1, 2, 2) table of dots.
scores = t.einsum('bqd,bkd->bqk', q, k)
assert scores.shape == (1, 2, 2)
assert scores[0].tolist() == [[1.0, 1.0],    # q0.k0, q0.k1
                              [0.0, 1.0]]    # q1.k0, q1.k1

# (Real attention: scale by 1/sqrt(d), softmax over k. Both live OUTSIDE
# einsum — they're not contractions.)
s = t.tensor([[[0.5, 0.5],
               [0.0, 1.0]]])               # a fake attention matrix (b,q,k)
v = t.tensor([[[10.0, 0.0],
               [0.0, 10.0]]])              # (b, k, d2=2)

# Mixing: k contracts -> each query gets its weighted sum of value rows.
out = t.einsum('bqk,bkd->bqd', s, v)
assert out[0, 0].tolist() == [5.0, 5.0]     # 0.5*v0 + 0.5*v1
assert out[0, 1].tolist() == [0.0, 10.0]    # 1.0*v1 — query 1 attends key 1

# Projection: one (d2, f) matrix shared across batch and sequence.
w = t.tensor([[1.0, 0.0, 0.0],
              [0.0, 1.0, 1.0]])            # (d2=2, f=3)
proj = t.einsum('bqd,df->bqf', out, w)
assert proj.shape == (1, 2, 3)
assert proj[0, 1].tolist() == [0.0, 10.0, 10.0]
print("bqd,bkd->bqk  scores", tuple(scores.shape))
print(scores[0])
print("bqk,bkd->bqd  mixed ", tuple(out.shape))
print(out[0])
print("bqd,df->bqf   proj  ", tuple(proj.shape))
print(proj[0])


<!-- dd:dd-q299 -->

### Problem 299 · faded

Single-head attention scores: all query-key dots.


In [ ]:
import torch as t

def solve(q, k):
    """(b,q,d) x (b,k,d) -> (b,q,k): d contracts, q/k combine."""
    return t.einsum('_____', q, k)


<!-- dd:dd-q263 -->

### Problem 263 · guided

Write a function solve(q, k) that takes Query and Key tensors of shapes (b, h, q, d) and (b, h, k, d), and returns the (b, h, q, k) ATTENTION SCORE tensor: for each batch and head, the dot product of every query vector with every key vector (no scaling).


<details>
<summary>Hints</summary>

1. Same scores with a HEADS axis: (b,h,q,d) and (b,h,k,d) — which letters
   are along for the ride?
2. b and h batch; d contracts; q, k combine.
3. `'bhqd,bhkd->bhqk'` — the multi-head score tensor.

</details>


In [ ]:
import torch as t

def solve(q, k):
    """Return the (b, h, q, k) ATTENTION SCORE tensor."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 1, 1, 2)), t.ones((1, 1, 2, 2))))


<!-- dd:dd-q254 -->

### Problem 254 · independent

Write a function solve(s, v) that takes attention weights s of shape (b, h, q, k) and values v of shape (b, h, k, d), and returns the (b, h, q, d) result of applying the weights: for each batch and head, the weighted sum over the k (key) axis of the value vectors.


In [ ]:
import torch as t

def solve(s, v):
    """Return the (b, h, q, d) result of applying the weights."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 1, 1, 2)) / 2, t.arange(4.0).reshape(1, 1, 2, 2)))


<!-- dd:dd-q301 -->

### Problem 301 · independent

Write a function solve(p, q) that takes two tensors of shape (b, h, n, d) and (b, h, m, d), and returns the (b, h, n, m) similarity tensor: within each batch item and head, the dot product of every vector in p with every vector in q.


In [ ]:
import torch as t

def solve(p, q):
    """Return the (b, h, n, m) similarity tensor."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 1, 1, 2)), t.ones((1, 1, 3, 2))))


<!-- dd:dd-q305 -->

### Problem 305 · independent

Write a function solve(h, w) that takes hidden states h of shape (b, t, hd) and an output weight matrix w of shape (hd, d), and returns the (b, t, d) tensor projecting every position's hidden vector through w — an MLP output layer over a sequence.


In [ ]:
import torch as t

def solve(h, w):
    """Return the (b, t, d) tensor projecting every position's hidden vector through w — an MLP output layer over a s"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 1, 2)), t.eye(2)))


<!-- dd:dd-q272 -->

### Problem 272 · independent

Write a function solve(x, m) that takes a sequence tensor x of shape (b, t, c) and a projection matrix m of shape (c, f), and returns the (b, t, f) tensor with the feature axis projected through m — every token's feature vector multiplied by the matrix.


In [ ]:
import torch as t

def solve(x, m):
    """Return the (b, t, f) tensor with the feature axis projected through m — every token's feature vector multiplie"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 2, 2)), t.eye(2)))


<!-- dd:dd-q288 -->

### Problem 288 · independent

Write a function solve(x, w) that takes input x of shape (b, t, d) and a stacked weight tensor w of shape (n, d, e) holding n projection matrices, and returns the (n, b, t, e) tensor applying ALL n projections to x simultaneously — projection k in output slot k.


In [ ]:
import torch as t

def solve(x, w):
    """Return the (n, b, t, e) tensor applying ALL n projections to x simultaneously — projection k in output slot k."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((1, 1, 2)), t.stack([t.eye(2), 2 * t.eye(2)])))


<!-- dd:dd-q283 -->

### Problem 283 · independent

Write a function solve(x, w1, w2) that takes input x of shape (b, i) and weight matrices w1 of shape (i, h) and w2 of shape (h, o), and returns the (b, o) result of pushing x through BOTH linear maps in one contraction (no intermediate activation).


In [ ]:
import torch as t

def solve(x, w1, w2):
    """Return the (b, o) result of pushing x through BOTH linear maps in one contraction (no intermediate activation)"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 0.0]]), t.eye(2), t.eye(2)))


#### Common mistakes

- **"Attention einsums are special DL operations."** — They're the pairwise
  table (scores) and the weighted sum (mixing) with batch letters attached.
  If you can read 'i,j->ij' and 'ij,j->i', these are the same rules at
  rank 4.
- **"Softmax/scaling can be folded into the spec."** — einsum is
  multiply-and-sum only. Scale (÷√d) and softmax happen between the two
  contractions, in ordinary PyTorch.
- **"The letters q and k are special keywords."** — Still just names —
  but GOOD names: matching the letters to the domain (batch, head, query,
  key, depth) is what makes five-axis specs readable. Adopt the convention;
  don't imagine the engine sees it.


<!-- dd:dd-kp-einsum-matrix-forms -->

## Matrix forms — quadratic, Gram, covariance

`einsum.matrix-forms`


Classical matrix expressions from statistics and geometry compress into
short specs — this KP is a tour of the ones the drills (and ML papers) use,
as translation practice between math notation and einsum:

- **Quadratic form** xᵀWx = Σᵢⱼ xᵢWᵢⱼxⱼ → `'i,ij,j->'` — three operands,
  every letter dropped: a fully contracted scalar. The spec is literally
  the double sum with the Σs removed.
- **Gram matrix** XᵀX for X of shape (n, d): entry [p, q] = column p ·
  column q = Σₙ XₙₚXₙₙq → `'nd,ne->de'` — the same tensor twice, the shared
  observation axis n contracted, the two FEATURE axes kept under different
  letters (d, e — same size, distinct roles).
- **Sample covariance**: the Gram of the COLUMN-CENTERED data, divided by
  n−1 — `t.einsum('nd,ne->de', xc, xc) / (n - 1)` where
  `xc = x - x.mean(axis=0)`. einsum does the contraction; centering
  (np-3) and the 1/(n−1) live outside — the familiar division-outside rule.
- **Pairwise row dots** between two sets: `'nd,md->nm'` — the linear-algebra
  core of similarity matrices (np-4's cosine, pre-normalization).
- **Batch aggregates**: Σₙ xₙxₙᵀ → `'bi,bj->ij'` (drop the batch letter);
  per-item quadratic forms vᵢᵀMvᵢ for a stack of vectors →
  `'bi,ij,bj->b'` — b kept, i and j contracted against a SHARED M.

The through-line: **repeated math indices = repeated einsum letters;
summation signs = letters missing from the output.** Any Σ-expression you
can write on paper transliterates directly. When you meet an unfamiliar
matrix identity, writing its einsum is often the fastest way to both
understand and implement it.


Task: a quadratic form, a Gram matrix, and a covariance — each checked
against its classical spelling.


In [ ]:
import torch as t

x = t.tensor([1.0, 2.0])
w = t.tensor([[3.0, 1.0],
              [0.0, 2.0]])

# x^T W x: the double sum, fully contracted.
qf = t.einsum('i,ij,j->', x, w, x)
assert qf == float(x @ w @ x)
assert qf == 13.0        # 1*3*1 + 1*1*2 + 2*0*1 + 2*2*2

# Gram matrix of columns: same tensor twice, observation axis contracted.
data = t.tensor([[1.0, 10.0],
                 [2.0, 20.0],
                 [3.0, 30.0]])          # (n=3, d=2)
gram = t.einsum('nd,ne->de', data, data)
assert t.allclose(gram, data.T @ data)
assert gram[0, 1] == 1*10 + 2*20 + 3*30    # col 0 . col 1

# Sample covariance: center columns first, contract, divide by n-1.
xc = data - data.mean(dim=0)
cov = t.einsum('nd,ne->de', xc, xc) / (data.shape[0] - 1)
assert t.allclose(cov, t.cov(data.T))
print("quadratic form x^T W x =", qf.item())
print("gram 'nd,ne->de'")
print(gram)
print("covariance")
print(cov)




Why each step:

1. Expanding the quadratic form by hand once (all four terms) demystifies
   the three-operand spec: each (i, j) pair contributes xᵢWᵢⱼxⱼ; einsum
   enumerates and sums them. The `x @ w @ x` twin confirms it.
2. In the Gram spec, the deliberate oddity is d vs e for axes of the SAME
   tensor: einsum needs distinct names for distinct roles (output row vs
   column), even when sizes coincide. 'nd,nd->dd' would instead walk a
   diagonal — wrong operation.
3. The covariance assembles three lessons — centering (np-3), the Gram
   contraction, division outside — and lands exactly on `t.cov`. Building
   library functions from primitives, then checking against the library, is
   the final form of validate-don't-assert.


In [ ]:
import torch as t

x = t.tensor([1.0, 2.0])
w = t.tensor([[3.0, 1.0],
              [0.0, 2.0]])

# x^T W x: the double sum, fully contracted.
qf = t.einsum('i,ij,j->', x, w, x)
assert qf == float(x @ w @ x)
assert qf == 13.0        # 1*3*1 + 1*1*2 + 2*0*1 + 2*2*2

# Gram matrix of columns: same tensor twice, observation axis contracted.
data = t.tensor([[1.0, 10.0],
                 [2.0, 20.0],
                 [3.0, 30.0]])          # (n=3, d=2)
gram = t.einsum('nd,ne->de', data, data)
assert t.allclose(gram, data.T @ data)
assert gram[0, 1] == 1*10 + 2*20 + 3*30    # col 0 . col 1

# Sample covariance: center columns first, contract, divide by n-1.
xc = data - data.mean(dim=0)
cov = t.einsum('nd,ne->de', xc, xc) / (data.shape[0] - 1)
assert t.allclose(cov, t.cov(data.T))
print("quadratic form x^T W x =", qf.item())
print("gram 'nd,ne->de'")
print(gram)
print("covariance")
print(cov)


<!-- dd:dd-q252 -->

### Problem 252 · faded

The quadratic form xᵀWx.


In [ ]:
import torch as t

def solve(x, w):
    """Sum_ij x_i W_ij x_j — fully contracted."""
    return t.einsum('_____', x, w, x)


<!-- dd:dd-q258 -->

### Problem 258 · guided

Write a function solve(x) that takes a 2-D tensor of shape (n, d) and returns the (d, d) matrix X^T X — the Gram matrix of x's COLUMNS, where entry [p, q] is the dot product of column p with column q.


<details>
<summary>Hints</summary>

1. XᵀX, entry [p, q] = column p dotted with column q — which axis is shared
   between the two copies of x, and which axes survive?
2. The two surviving axes need DIFFERENT letters even though both come from
   x's column axis.
3. `'nd,ne->de'` — check one off-diagonal entry against a hand dot.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the (d, d) matrix X^T X — the Gram matrix of x's COLUMNS, where entry [p, q] is the dot product of colu"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]])))


<!-- dd:dd-q259 -->

### Problem 259 · independent

Write a function solve(x) that takes a 2-D tensor of shape (n, d) with n >= 2 rows of observations and returns the (d, d) SAMPLE COVARIANCE matrix: center each column by its mean, contract the observation axis between the centered matrix and itself, and divide by n - 1.


In [ ]:
import torch as t

def solve(x):
    """Return the (d, d) SAMPLE COVARIANCE matrix."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 6.0]])))


<!-- dd:dd-q248 -->

### Problem 248 · independent

Write a function solve(x, y) that takes two 2-D float tensors x of shape (n, d) and y of shape (m, d), and returns the (n, m) matrix of ALL pairwise row dot products: entry [i, j] is the dot product of row i of x with row j of y. This is the Gram-style product x @ y.T, written as one contraction.


In [ ]:
import torch as t

def solve(x, y):
    """Return the (n, m) matrix of dot products between rows of x and y."""
    return None


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([[1.0, 0.0]]), t.tensor([[2.0, 3.0], [0.0, 5.0]])))


<!-- dd:dd-q306 -->

### Problem 306 · independent

Write a function solve(v, m) that takes a BATCH of vectors v of shape (b, n) and a single matrix m of shape (n, n), and returns the length-b vector of QUADRATIC FORMS: entry k is v[k] @ m @ v[k]. (A companion drill does one vector; the batch letter is what's new.)


In [ ]:
import torch as t

def solve(v, m):
    """Return the length-b vector of QUADRATIC FORMS."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0]]), t.eye(2)))


#### Common mistakes

- **"Two axes of the same size should share a letter."** — Letters encode
  ROLE, not size. The Gram's output row and column both come from x's
  feature axis but must be d and e; sharing the letter would compute a
  diagonal instead. Size-match is necessary for sharing, never sufficient.
- **"Covariance is an einsum one-liner."** — The CONTRACTION is; centering
  and the 1/(n−1) are not (einsum neither subtracts nor divides). Three
  short lines, each doing one thing.
- **"Math-to-einsum translation is ad hoc."** — It's mechanical: subscripts
  become letters, Σ over an index = omit that letter from the output,
  independent products = separate operands. Practice the transliteration
  direction math→spec; the reverse (reading) then comes free.
